# Unified Session Preprocessing (SR-GNN/TAGNN-Compatible)

This notebook prepares clean session sequences with exact filtering logic used by SR-GNN/TAGNN, but stores one unified artifact per dataset.

Raw files used:
- `data/raw/yoochoose/yoochoose-clicks.pkl.gz`
- `data/raw/diginetica/train-item-views.pkl.gz`

Ignored files:
- Yoochoose buys file
- Diginetica purchases file
- official challenge test files
- product/category/metadata files

Outputs (single compressed artifact per dataset):
- `processed/yoochoose_sessions.pkl.gz`
- `processed/diginetica_sessions.pkl.gz`

Note:
- No train/test split is persisted at file level.
- Future notebooks should load these processed files and perform split/vocabulary logic in-code.

In [ ]:
import gzip
import pickle
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

YOOCHOOSE_RAW = RAW_DIR / "yoochoose" / "yoochoose-clicks.pkl.gz"
DIGINETICA_RAW = RAW_DIR / "diginetica" / "train-item-views.pkl.gz"

print(f"Yoochoose compressed raw file exists: {YOOCHOOSE_RAW.exists()}")
print(f"Diginetica compressed raw file exists: {DIGINETICA_RAW.exists()}")

Yoochoose compressed raw file exists: True
Diginetica compressed raw file exists: True


In [ ]:
def iter_chunked_pickle(path: Path):
    """Yield DataFrame chunks from compressed raw artifact."""
    with gzip.open(path, "rb") as f:
        metadata = pickle.load(f)  # first object
        if not isinstance(metadata, dict):
            raise ValueError(f"Invalid metadata in {path}")

        while True:
            try:
                yield pickle.load(f)
            except EOFError:
                break


def load_yoochoose_sessions(path: Path) -> list[tuple[int, pd.Timestamp, list[int]]]:
    """Build ordered Yoochoose sessions from chunked compressed raw file."""
    events_by_session: dict[int, list[tuple[pd.Timestamp, int]]] = defaultdict(list)

    for chunk in iter_chunked_pickle(path):
        chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], utc=True).dt.tz_convert(
            None
        )

        for sid, part in chunk.groupby("session_id", sort=False):
            times = part["timestamp"].tolist()
            items = part["item_id"].astype("int64").tolist()
            events_by_session[int(sid)].extend(zip(times, items))

    sessions: list[tuple[int, pd.Timestamp, list[int]]] = []
    for sid, events in events_by_session.items():
        events.sort(key=lambda x: x[0])
        items = [item for _, item in events]
        session_date = events[-1][0].normalize()
        sessions.append((sid, session_date, items))

    return sessions


def load_diginetica_sessions(path: Path) -> list[tuple[int, pd.Timestamp, list[int]]]:
    """Build ordered Diginetica sessions from chunked compressed raw file."""
    events_by_session: dict[int, list[tuple[int, int]]] = defaultdict(list)
    session_dates: dict[int, pd.Timestamp] = {}

    for chunk in iter_chunked_pickle(path):
        chunk["eventdate"] = pd.to_datetime(chunk["eventdate"], format="%Y-%m-%d")

        for sid, part in chunk.groupby("session_id", sort=False):
            sid = int(sid)
            frames = part["timeframe"].astype("int64").tolist()
            items = part["item_id"].astype("int64").tolist()
            events_by_session[sid].extend(zip(frames, items))

            event_date = pd.Timestamp(part["eventdate"].max()).normalize()
            prev_date = session_dates.get(sid)
            session_dates[sid] = (
                event_date if prev_date is None else max(prev_date, event_date)
            )

    sessions: list[tuple[int, pd.Timestamp, list[int]]] = []
    for sid, events in events_by_session.items():
        events.sort(key=lambda x: x[0])
        items = [item for _, item in events]
        session_date = session_dates[sid]
        sessions.append((sid, session_date, items))

    return sessions


def filter_sessions(
    sessions: list[tuple[int, pd.Timestamp, list[int]]], min_item_freq: int = 5
) -> list[tuple[int, pd.Timestamp, list[int]]]:
    """Apply SR-GNN filtering order: remove len=1, count item frequency, remove item<5, remove len<2."""
    non_trivial_sessions = [
        (sid, date, items) for sid, date, items in sessions if len(items) > 1
    ]

    item_frequency = Counter()
    for _, _, items in non_trivial_sessions:
        item_frequency.update(items)

    filtered_sessions: list[tuple[int, pd.Timestamp, list[int]]] = []
    for sid, date, items in non_trivial_sessions:
        kept_items = [item for item in items if item_frequency[item] >= min_item_freq]
        if len(kept_items) >= 2:
            filtered_sessions.append((sid, date, kept_items))

    return filtered_sessions


def sort_sessions_by_date(
    sessions: list[tuple[int, pd.Timestamp, list[int]]],
) -> list[tuple[int, pd.Timestamp, list[int]]]:
    """Sort sessions chronologically (date, then session id for stable tie-break)."""
    return sorted(sessions, key=lambda x: (x[1], x[0]))


def session_stats(
    sessions: list[tuple[int, pd.Timestamp, list[int]]],
) -> dict[str, float]:
    clicks = sum(len(items) for _, _, items in sessions)
    session_count = len(sessions)
    unique_items = len({item for _, _, items in sessions for item in items})
    avg_len = (clicks / session_count) if session_count else 0.0
    return {
        "clicks": clicks,
        "sessions": session_count,
        "items": unique_items,
        "avg_len": avg_len,
    }


def save_processed_dataset(
    output_path: Path,
    dataset_name: str,
    sessions: list[tuple[int, pd.Timestamp, list[int]]],
) -> None:
    """Save one compressed artifact per dataset for downstream notebooks."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "dataset": dataset_name,
        "session_ids": [sid for sid, _, _ in sessions],
        "session_dates": [date.strftime("%Y-%m-%d") for _, date, _ in sessions],
        "session_item_sequences": [items for _, _, items in sessions],
        "stats": session_stats(sessions),
    }

    with gzip.open(output_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)


def print_stats(name: str, stats: dict[str, float]) -> None:
    print(
        f"{name:16s} | clicks={stats['clicks']:,} | sessions={stats['sessions']:,} | "
        f"items={stats['items']:,} | avg_len={stats['avg_len']:.2f}"
    )

## Unified preprocessing function

Function below executes shared preprocessing pipeline and stores one processed dataset artifact:
1. Read compressed raw chunks (`.pkl.gz`) generated in download notebook.
2. Reconstruct ordered sessions.
3. Apply filtering order (`len==1` removal first, then item freq filter `<5`).
4. Sort sessions chronologically.
5. Persist one compressed file with session ids, dates, and item sequences.

No train/test split written to disk in this notebook.

In [ ]:
def preprocess_dataset(
    name: str,
    raw_path: Path,
    session_loader,
):
    print(f"\n===== {name.upper()} =====")
    print(f"Reading compressed file...")

    if not raw_path.exists():
        raise FileNotFoundError(
            f"Missing input file: {raw_path}. Run dataset_download.ipynb first."
        )

    sessions = session_loader(raw_path)
    filtered_sessions = filter_sessions(sessions, min_item_freq=5)
    sorted_sessions = sort_sessions_by_date(filtered_sessions)

    stats = session_stats(sorted_sessions)
    print("After filtering (step 6 sanity checkpoint):")
    print_stats(f"{name} filtered", stats)

    return {
        "name": name,
        "sessions": sorted_sessions,
        "stats": stats,
    }

In [21]:
yoochoose_data = preprocess_dataset(
    name="yoochoose",
    raw_path=YOOCHOOSE_RAW,
    session_loader=load_yoochoose_sessions,
)

diginetica_data = preprocess_dataset(
    name="diginetica",
    raw_path=DIGINETICA_RAW,
    session_loader=load_diginetica_sessions,
)

PROCESSED_DIR = OUTPUT_ROOT / "processed"

yoochoose_out = PROCESSED_DIR / "yoochoose_sessions.pkl.gz"
diginetica_out = PROCESSED_DIR / "diginetica_sessions.pkl.gz"

save_processed_dataset(
    output_path=yoochoose_out,
    dataset_name="yoochoose",
    sessions=yoochoose_data["sessions"],
)

save_processed_dataset(
    output_path=diginetica_out,
    dataset_name="diginetica",
    sessions=diginetica_data["sessions"],
)

print("Saved unified processed artifacts")


===== YOOCHOOSE =====
Reading compressed file...
After filtering (step 6 sanity checkpoint):
yoochoose filtered | clicks=31,708,505 | sessions=7,981,581 | items=37,486 | avg_len=3.97

===== DIGINETICA =====
Reading compressed file...
After filtering (step 6 sanity checkpoint):
diginetica filtered | clicks=993,483 | sessions=204,789 | items=43,136 | avg_len=4.85
Saved unified processed artifacts


In [22]:
step6_target = pd.DataFrame(
    [
        {"dataset": "yoochoose", "sessions": 7_981_580, "items": 37_483},
        {"dataset": "diginetica", "sessions": 204_771, "items": 43_097},
    ]
).set_index("dataset")

step6_actual = pd.DataFrame(
    [
        {
            "dataset": "yoochoose",
            "sessions": int(yoochoose_data["stats"]["sessions"]),
            "items": int(yoochoose_data["stats"]["items"]),
        },
        {
            "dataset": "diginetica",
            "sessions": int(diginetica_data["stats"]["sessions"]),
            "items": int(diginetica_data["stats"]["items"]),
        },
    ]
).set_index("dataset")

print("Step 6 sanity check (actual vs reference):")
display(pd.concat({"actual": step6_actual, "target": step6_target}, axis=1))

Step 6 sanity check (actual vs reference):


actual          target       
           sessions  items sessions  items
dataset                                   
yoochoose   7981581  37486  7981580  37483
diginetica   204789  43136   204771  43097

In [23]:
with gzip.open(yoochoose_out, "rb") as f:
    yoochoose_payload = pickle.load(f)

with gzip.open(diginetica_out, "rb") as f:
    diginetica_payload = pickle.load(f)

print("Payload keys:", sorted(yoochoose_payload.keys()))
print("Yoochoose sessions stored:", len(yoochoose_payload["session_item_sequences"]))
print("Diginetica sessions stored:", len(diginetica_payload["session_item_sequences"]))
print("Example Yoochoose session:", yoochoose_payload["session_item_sequences"][0][:10])

Payload keys: ['dataset', 'recommended_split_days', 'session_dates', 'session_ids', 'session_item_sequences', 'stats']
Yoochoose sessions stored: 7981581
Diginetica sessions stored: 204789
Example Yoochoose session: [214577732, 214587013, 214577732]
